<a href="https://colab.research.google.com/github/kjahan/nlp_tips_and_trick/blob/main/notebooks/TwinBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TwinBERT

https://github.com/deepampatel/TwinBert

https://arxiv.org/pdf/2002.06275.pdf

## Install transformers v3

There was an issue with latest version of transforners so had to install older version:

https://github.com/huggingface/transformers/issues/2478

In [17]:
!pip install transformers==3

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!pip freeze | grep transformers

transformers==3.0.0


## Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn import metrics

import transformers
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertTokenizer, BertModel, BertConfig
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

## Load Quora dataset

The dataset is Quora questions pairs dataset. Ideally in the paper the authors have trained the model to use it as a backend for a sponsored search engine, to delivers ads alongside the organic search results.

https://www.kaggle.com/competitions/quora-question-pairs/data

Download train.csv.zip and test.csv.zip manually to Google colab data folder!

In [ ]:
!mkdir data

In [ ]:
!unzip data/train.csv.zip -d data/
#!unzip data/test.csv.zip data/

Archive:  data/train.csv.zip
  inflating: data/train.csv          


In [ ]:
df = pd.read_csv("data/train.csv")   # Dataset : https://www.kaggle.com/c/quora-question-pairs

In [ ]:
df.head(5)

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [19]:
df.shape

(404290, 6)

## Dataset Loader for the Siamese Network

In [20]:
class SiameseNetworkDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.question1 = dataframe.question1
        self.question2 = dataframe.question2
        self.targets = dataframe.is_duplicate
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
    
    
    def tokenize(self,input_text):
        input_text = " ".join(input_text.split())

        inputs = self.tokenizer.encode_plus(
            input_text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            pad_to_max_length=True,
            truncation=True,
            return_token_type_ids=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]
        return ids,mask,token_type_ids

    def __getitem__(self, index):
        ids1,mask1,token_type_ids1 = self.tokenize(str(self.question1[index]))
        ids2,mask2,token_type_ids2 = self.tokenize(str(self.question2[index]))
        


        return {
            'ids': [torch.tensor(ids1, dtype=torch.long),torch.tensor(ids2, dtype=torch.long)],
            'mask': [torch.tensor(mask1, dtype=torch.long),torch.tensor(mask2, dtype=torch.long)],
            'token_type_ids': [torch.tensor(token_type_ids1, dtype=torch.long),torch.tensor(token_type_ids2, dtype=torch.long)],
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }
   

## TwinBert architecture


In [21]:
class TwinBert(nn.Module):
    def __init__(self):
        super(TwinBert, self).__init__()
        self.model = transformers.BertModel.from_pretrained('bert-base-uncased')

    def forward_once(self, ids, mask, token_type_ids):
        _, output= self.model(ids, attention_mask = mask, token_type_ids = token_type_ids)
        return output

    def forward(self, ids, mask, token_type_ids):
        output1 = self.forward_once(ids[0],mask[0], token_type_ids[0])
        output2 = self.forward_once(ids[1],mask[1], token_type_ids[1])
        return output1, output2

# Just right before the actual usage
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device: {}".format(device))
model = TwinBert()
model.to(device)

Device: cuda


TwinBert(
  (model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  

In [22]:
MAX_LEN = 200
TRAIN_BATCH_SIZE = 8
VALID_BATCH_SIZE = 4
EPOCHS = 1
LEARNING_RATE = 1e-05
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

## Downsample

In [27]:
n = 20000
df = df.head(n)

In [28]:
train_size = 0.8
train_dataset=df.sample(frac=train_size,random_state=200).reset_index(drop=True)
test_dataset=df.drop(train_dataset.index).reset_index(drop=True)


print("FULL Dataset: {}".format(df.shape))
print("TRAIN Dataset: {}".format(train_dataset.shape))
print("TEST Dataset: {}".format(test_dataset.shape))

training_set = SiameseNetworkDataset(train_dataset, tokenizer, MAX_LEN)
testing_set = SiameseNetworkDataset(test_dataset, tokenizer, MAX_LEN)

FULL Dataset: (20000, 6)
TRAIN Dataset: (16000, 6)
TEST Dataset: (4000, 6)


In [29]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

## Loss Function

A contrastive loss function that takes cosine similarity as a metric to measure the distance.


In [30]:
class CosineContrastiveLoss(nn.Module):
    def __init__(self, margin=0.4):
        super(CosineContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        cos_sim = F.cosine_similarity(output1, output2)
        loss_cos_con = torch.mean((1-label) * torch.div(torch.pow((1.0-cos_sim), 2), 4) +
                                    (label) * torch.pow(cos_sim * torch.lt(cos_sim, self.margin), 2))
        return loss_cos_con

In [31]:
criterion = CosineContrastiveLoss()
optimizer = optim.Adam(model.parameters(),lr = 0.0005 )

## Training

In [32]:
def train(epoch):
    model.train()
    for _,data in enumerate(training_loader, 0):
        ids,mask,token_type_ids = data['ids'],data['mask'],data['token_type_ids'] 
        targets = data['targets'].to(device, dtype = torch.float)
        ids = [ids[0].to(device, dtype = torch.long),ids[1].to(device, dtype = torch.long)]
        mask = [mask[0].to(device, dtype = torch.long),mask[1].to(device, dtype = torch.long)]
        token_type_ids = [token_type_ids[0].to(device, dtype = torch.long),token_type_ids[1].to(device, dtype = torch.long)]
        output1,output2 = model(ids, mask, token_type_ids)
        optimizer.zero_grad()
        loss = criterion(output1,output2,targets)
        if _%5000==0:
            print(f'Step: {_}, Epoch: {epoch}, Loss:  {loss.item()}')
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [33]:
for epoch in range(EPOCHS):
    train(epoch)

Step: 0, Epoch: 0, Loss:  0.000600535306148231


## Validation

In [36]:
def validation():
    model.eval()
    fin_targets=[]
    fin_outputs=[]
    with torch.no_grad():
        for _, data in enumerate(testing_loader, 0):
            ids,mask,token_type_ids = data['ids'],data['mask'],data['token_type_ids'] 
            targets = data['targets'].to(device, dtype = torch.float)
            ids = [ids[0].to(device, dtype = torch.long),ids[1].to(device, dtype = torch.long)]
            mask = [mask[0].to(device, dtype = torch.long),mask[1].to(device, dtype = torch.long)]
            token_type_ids = [token_type_ids[0].to(device, dtype = torch.long),token_type_ids[1].to(device, dtype = torch.long)]
            targets = data['targets'].to(device, dtype = torch.float)
            output1,output2 = model(ids, mask, token_type_ids)
            cos_sim = F.cosine_similarity(output1, output2)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(cos_sim).cpu().detach().numpy().tolist())
    return fin_outputs, fin_targets


In [37]:
outputs, targets = validation()
outputs = np.array(outputs) >= 0.5
accuracy = metrics.accuracy_score(targets, outputs)
f1_score_micro = metrics.f1_score(targets, outputs, average='micro')
f1_score_macro = metrics.f1_score(targets, outputs, average='macro')
print(f"Accuracy Score = {accuracy}")
print(f"F1 Score (Micro) = {f1_score_micro}")
print(f"F1 Score (Macro) = {f1_score_macro}")

Accuracy Score = 0.37225
F1 Score (Micro) = 0.37224999999999997
F1 Score (Macro) = 0.2712698123519767
